# nanoTSFM on Runpod

Run the toy check, train the full baseline, plot a forecast and obtain its GIFT-Eval score.
This adapts `quick_start.ipynb` for one A100 80GB GPU without shortening the baseline.

Deploy the template linked in [the course guide](docs/course.md#run-on-runpod), open Jupyter,
and select **nanoTSFM (Python 3.13)**. Run the cells in order. Setup and downloads are paid pod time;
terminate the pod after downloading any results you need.

## Setup

The template has already run setup. This cell verifies the environment and is also safe to run
again. Commands stop on failure. Their complete wall times include process startup and any downloads.

In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "run.sh").is_file(), "Open this notebook from the nanoTSFM repository root."
os.environ["TORCH"] = "cuda"
timings = {}


def run(*args):
    started = time.perf_counter()
    command = ["./run.sh", *args]
    with subprocess.Popen(
        command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    timings[" ".join(command)] = time.perf_counter() - started
    print(f"Wall time: {timings[' '.join(command)]:.1f} s")


run("setup")
import torch

assert torch.cuda.is_available(), "This notebook requires a GPU pod and the nanoTSFM kernel."
print(torch.cuda.get_device_name(0))

## Toy check

This small CPU run checks that training works before downloading the real data.

In [ ]:
run("toy")

## Data and full baseline

Download GEP-M and the held-out diagnostics. Train the unchanged `configs/baseline.yaml`:
5,000 steps with seed 7, followed by GEP-Val. Rerunning the training cell writes to the same
`runs/baseline` directory.

In [ ]:
run("data")

In [ ]:
run("train", "baseline")
run_dir = Path(os.environ.get("RUNS", "runs")) / "baseline"
training = json.loads((run_dir / "run.json").read_text())
print(f"{training['steps']} steps, {training['elapsed_seconds']:.1f} training seconds")

## Forecast

As in `quick_start.ipynb`, plot one held-out series. The line is the median forecast; the band
spans the 10th to 90th percentiles.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from nanotsfm.data import arrays, load
from nanotsfm.model import forecast, load_checkpoint

model, _ = load_checkpoint(run_dir / "checkpoint.pt", "cuda")
context, horizon = model.config.context_length, model.config.prediction_length
validation = load("GEP-M", "validation")
index, values = next(
    (i, v[0])
    for i, v in enumerate(arrays(validation))
    if v.shape[0] == 1 and v.shape[1] >= context + horizon
)
history, future = values[-(context + horizon) : -horizon], values[-horizon:]
quantiles = forecast(model, torch.tensor(history)[None], horizon)[0].cpu().numpy()

steps = np.arange(-len(history), horizon)
plt.figure(figsize=(10, 3))
plt.plot(steps[: len(history)], history, color="0.4", lw=0.8, label="history")
plt.plot(steps[len(history) :], future, color="black", lw=1.2, label="actual")
plt.fill_between(steps[len(history) :], quantiles[:, 0], quantiles[:, 8], alpha=0.3, label="10-90%")
plt.plot(steps[len(history) :], quantiles[:, 4], lw=1.2, label="median")
plt.title(validation[index]["item_id"], loc="left", fontsize=10)
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()

## GIFT-Eval score

Evaluate all 97 tasks from 23 datasets. The first run downloads the benchmark and its upstream code.
Relative CRPS below 1 beats Seasonal Naive; lower is better. The published baseline averages about
0.670 across three seeds, so a single run need not match that mean exactly.

In [ ]:
run("eval", "baseline")

In [ ]:
for label, filename in [("GEP-Val", "gep-val.json"), ("GIFT-Eval", "gift.json")]:
    result = json.loads((run_dir / filename).read_text())
    score = result["summary"]
    print(
        f"{label}: relative CRPS {score['geometric_relative_crps']:.4f}, "
        f"relative MASE {score['geometric_relative_mase']:.4f}, {len(result['rows'])} tasks"
    )

for command, seconds in timings.items():
    print(f"{seconds:8.1f} s  {command}")

## Save results and finish

Download `runs/baseline` from Jupyter's file browser before terminating the pod. It contains the
checkpoint, training log and scores. Downloading the files individually or making an archive in
Jupyter's terminal both work.

**Terminate the pod in Runpod when finished.** Stopping ends GPU charges but retains paid storage;
termination deletes the pod's files. See [the course guide](docs/course.md#run-on-runpod) for measured
costs, [directions](docs/directions.md) for experiments, and [submission](docs/submission.md) for the
three-seed submission procedure.